## 01 - Load dataset to unified format
Load a dataset slice via the registry, inspect the first sample, and persist all samples into the repository’s unified JSONL schema (one QA sample per line).

In [2]:
# Robustly set repo root and import path
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    if (cand / "src").exists():
        ROOT = cand
        break
else:
    raise RuntimeError("Could not locate repo root containing src/.")

# Add repo root so imports like 'src.data_loader' resolve
sys.path.append(str(ROOT))
print(f"Repo root: {ROOT}")

Repo root: C:\uni\chunks\Chunking-Research


### Configure what to load and where to store it
Adjust dataset, split/limit for speed, and the unified output path.

In [3]:
# Configure dataset slice and output
# Choose any registered dataset name; adjust split/limit to keep runs quick.
dataset_name = 'poquad'  # registered in src/data_loader/datasets
load_kwargs = {
    'split': 'train',
    # Use parquet-converted revision to avoid dataset scripts and reduce deps
    'revision': 'refs/convert/parquet',
    'cache_dir': str(ROOT / 'data/hf_cache'),
    'limit': 1000,
}

output_path = ROOT / 'data/processed/demo_samples.jsonl'  # unified format destination


### Load the dataset slice via registry and preview
This uses the dataset loader registered under `dataset_name`, then prints the first sample.

In [4]:
# Load dataset slice via registry and preview first sample
from src.data_loader import get_dataset_loader, save_samples_jsonl

loader = get_dataset_loader(dataset_name)
samples = loader(**load_kwargs)
print(f'Loaded {len(samples)} samples')
print('First sample:')
print(samples[0].to_dict())

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loaded 1000 samples
First sample:
{'sample_id': '1', 'context': 'Projekty konfederacji zaczęły się załamywać 5 sierpnia 1942. Ponownie wróciła kwestia monachijska, co uaktywniło się wymianą listów Ripka – Stroński. Natomiast 17 sierpnia 1942 doszło do spotkania E. Beneša i J. Masaryka z jednej a Wł. Sikorskiego i E. Raczyńskiego z drugiej strony. Polscy dyplomaci zaproponowali podpisanie układu konfederacyjnego. W następnym miesiącu, tj. 24 września, strona polska przesłała na ręce J. Masaryka projekt deklaracji o przyszłej konfederacji obu państw. Strona czechosłowacka projekt przyjęła, lecz już w listopadzie 1942 E. Beneš podważył ideę konfederacji. W zamian zaproponowano zawarcie układu sojuszniczego z Polską na 20 lat (formalnie nastąpiło to 20 listopada 1942).', 'question': 'Co było powodem powrócenia konceptu porozumieniu monachijskiego?', 'answers': ['wymianą listów Ripka – Stroński'], 'answer_starts': [117], 'title': 'Konfederacja polsko-czechosłowacka'}


### Persist to unified JSONL
Writes one QA sample per line to reuse in later notebooks (chunking and evaluation).

In [5]:
# Save to unified JSONL (one sample per line)
save_samples_jsonl(samples, str(output_path))
print(f'Saved unified samples to {output_path}')

Saved unified samples to C:\uni\chunks\Chunking-Research\data\processed\demo_samples.jsonl
